<a href="https://colab.research.google.com/github/boyerb/Investments/blob/master/Ex08-AlphaBeta_WRDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Investments: Theory, Fundamental Analysis, and Data Driven Analytics**, Bates, Boyer, and Fletcher

# Example Chapter 8: Estimating Alpha Using Data from WRDS API  
  
In this example we estimate the alpha and beta of Johnson and Johnson (`ticker:JNJ`) relative to the market portfolio along with 95% confidence intervals. In this example, data for Johnson and Johnson returns come from the WRDS API. Data for the market and short-term T-bill returns come from [Ken French's website](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html). An introduction to the WRDS API is given in [EX07-WRDS.ipynb](https://colab.research.google.com/github/boyerb/Investments/blob/master/Ex07-WRDS.ipynb).  

### Imports and Setup
We first import a variety of useful packages. To improve readability when printing a DataFrame to the consol, we then configure Pandas display options: (1) show all columns without truncation, (2) widen the output to 1000 characters so rows print on a single line, and (3) truncate long tables.


In [ ]:
# Install the current farms package and WRDS client.
%pip install -q --upgrade farms wrds

import farms as fm
import pandas as pd
import wrds

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_rows', 20)


This notebook uses a WRDS connection to load monthly CRSP data for JNJ. The current `farms` loader also adds the market excess return and risk-free rate so the regression can use the same interface as the Alpha Vantage example.

In [ ]:
db = wrds.Connection()

df = fm.load_crsp_data(
    db=db,
    identifiers=["JNJ"],
    identifier_type="ticker",
    start_date="2015-01",
    end_date="2024-12",
    frequency="monthly",
    include_factors="market",
)

df = df.dropna(subset=["ret", "mkt-rf", "rf"]).copy()
df


### Load Returns for the Market and Short-Term T-Bills
This function retrieves monthly data for the Fama–French Five-Factor model from [Ken French's website](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html). From the resulting DataFrame, we will extract the market’s excess return `Mkt-RF` and the return on short-term T-bills `RF`.


In [ ]:
# The farms loader added the market excess return and risk-free rate.
df[["ret", "mkt-rf", "rf"]]


### Merge Two Data Sources
In this code we merge the data from Alpha Vantage and Ken French's website together. We then create the excess return for Johnson & Johnson.

In [ ]:
merged = df[["ret", "mkt-rf", "rf"]].copy()
merged["Ret-RF"] = merged["ret"] - merged["rf"]
merged


### Estimate Alpha, Beta, and 95% Confidence Intervals
Here we estimate the alpha and beta of Johnson & Johnson relative to the market portfolio along with the 95% confidence intervals.

In [ ]:
results = fm.run_ols(merged["mkt-rf"], merged["Ret-RF"])
results
